# CausalMan: Observational Dataset Generation Across Seeds

This notebook is based on `example_observational.ipynb`. It generates one
observational dataset for every seed in `SEEDS`, retaining exactly `N_SAMPLES`
rows for each dataset.

Each seed gets its own output directory. The notebook also stores the observable
variable list and projected ground-truth ADMG/MAG for each seed.

## 1. Configuration

Edit only this cell for a standard generation run.

In [ ]:
# ── The only cell you normally need to edit ──────────────────────────────────

CHOICE = "causalman_micro"       # micro | small | medium | large
SEEDS = [4, 6, 42, 66, 90]      # one dataset is generated for each seed
N_SAMPLES = 10_000               # rows retained in each observational dataset
BATCH_MULTIPLIER = 1             # increase if fewer than N_SAMPLES are generated

OUTPUT_ROOT = "output/causalman_observational"
PARALLELIZE = True
MAX_WORKERS = 5
DEBUG_MODE = False

# ─────────────────────────────────────────────────────────────────────────────

## 2. Imports and output directory

In [ ]:
import os
from datetime import datetime
from pathlib import Path
import sys

# Put the repository root before this notebook directory so causalman.py
# cannot shadow the causalman package when the notebook runs in-place.
PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
        and (path / "causalman" / "__init__.py").is_file()
    ),
    None,
)
if PROJECT_ROOT is not None:
    project_root = str(PROJECT_ROOT)
    if project_root in sys.path:
        sys.path.remove(project_root)
    sys.path.insert(0, project_root)

from causalman import CausalMan
from graph_projections import (
    get_latent_projection_single as latent_projection,
    count_edge_types,
    write_mixed_graph_graphml,
    admg2mag,
    validate_mag,
)

timestamp = datetime.now().strftime("%Y_%m_%d_%H%M%S")
run_dir = os.path.join(OUTPUT_ROOT, f"{CHOICE}_{timestamp}")
os.makedirs(run_dir, exist_ok=True)

print(f"Results will be saved to: {os.path.abspath(run_dir)}")

## 3. Generate one observational dataset per seed

For every seed, the notebook:

1. runs CausalMan;
2. keeps exactly `N_SAMPLES` rows using deterministic sampling;
3. marks constant columns as non-observable for `causalman_micro`, matching the
   original observational example;
4. saves the observable data and variable list;
5. creates the projected ADMG and MAG.

In [ ]:
generation_summary = []

for seed in SEEDS:
    print(f"\n── {CHOICE} | seed={seed} ──")

    seed_dir = os.path.join(run_dir, f"seed_{seed:03d}")
    simulator_dir = os.path.join(seed_dir, "simulator_output")
    os.makedirs(simulator_dir, exist_ok=True)

    simulator = CausalMan(
        name=CHOICE,
        seed=seed,
        batch_multiplier=BATCH_MULTIPLIER,
        parallelize=PARALLELIZE,
        max_workers=MAX_WORKERS,
        debug_mode=DEBUG_MODE,
        save_path=simulator_dir,
    )

    obs_dataset, all_interventional_tables, all_paths_df, dag_level_2 = (
        simulator.sample()
    )

    n_generated = len(obs_dataset)
    if n_generated < N_SAMPLES:
        raise RuntimeError(
            f"Seed {seed}: simulator generated {n_generated:,} rows, "
            f"but N_SAMPLES={N_SAMPLES:,}. Increase BATCH_MULTIPLIER "
            "or reduce N_SAMPLES."
        )

    sampled_dataset = (
        obs_dataset.sample(n=N_SAMPLES, random_state=seed)
        .reset_index(drop=True)
    )

    # Match the behavior of example_observational.ipynb for the micro scenario:
    # constant columns are treated as hidden before latent projection.
    if CHOICE == "causalman_micro":
        constant_columns = [
            column
            for column in sampled_dataset.columns
            if sampled_dataset[column].nunique(dropna=False) <= 1
        ]
        for column in constant_columns:
            if dag_level_2.has_node(column):
                dag_level_2.nodes[column]["Observable"] = False
    else:
        constant_columns = []

    observable_nodes = []
    hidden_nodes = []
    for node, attrs in dag_level_2.nodes(data=True):
        if "Observable" not in attrs or type(attrs["Observable"]) is not bool:
            raise ValueError(
                f"Seed {seed}: node {node!r} is missing a Boolean "
                "'Observable' attribute."
            )
        if attrs["Observable"]:
            observable_nodes.append(node)
        else:
            hidden_nodes.append(node)

    missing_columns = [
        node for node in observable_nodes if node not in sampled_dataset.columns
    ]
    if missing_columns:
        raise ValueError(
            f"Seed {seed}: observable graph nodes missing from the dataset: "
            f"{missing_columns}"
        )

    observable_df = sampled_dataset[observable_nodes]

    dataset_path = os.path.join(
        seed_dir,
        f"{CHOICE}_observational_n{N_SAMPLES}_RS{seed}.csv",
    )
    observable_df.to_csv(dataset_path, index=False)

    variables_path = os.path.join(seed_dir, "observable_variables.txt")
    with open(variables_path, "w", encoding="utf-8") as file:
        file.write("\n".join(observable_nodes) + "\n")

    interventional_table_path = os.path.join(
        seed_dir, f"{CHOICE}_interventional_table.csv"
    )
    all_interventional_tables.to_csv(interventional_table_path, index=False)

    # Latent projection to the observable variables.
    projected_admg = latent_projection(dag_level_2.copy())
    directed_count, bidirected_count = count_edge_types(projected_admg)
    admg_path = os.path.join(seed_dir, "projected_ground_truth_admg.graphml")
    write_mixed_graph_graphml(projected_admg, admg_path)

    mag = admg2mag(projected_admg)
    validate_mag(mag)
    mag_path = os.path.join(seed_dir, "projected_ground_truth_mag.graphml")
    write_mixed_graph_graphml(mag, mag_path)

    generation_summary.append(
        {
            "seed": seed,
            "generated_rows": n_generated,
            "saved_rows": len(observable_df),
            "observable_variables": len(observable_nodes),
            "hidden_variables": len(hidden_nodes),
            "constant_columns_hidden": len(constant_columns),
            "admg_directed_edges": directed_count,
            "admg_bidirected_edges": bidirected_count,
            "dataset_path": dataset_path,
        }
    )

    print(f"Generated rows: {n_generated:,}")
    print(f"Saved rows:     {len(observable_df):,}")
    print(f"Observables:    {len(observable_nodes)}")
    print(f"Dataset:        {dataset_path}")

## 4. Generation summary

In [ ]:
import pandas as pd

summary_df = pd.DataFrame(generation_summary)
display(summary_df)

summary_path = os.path.join(run_dir, "generation_summary.csv")
summary_df.to_csv(summary_path, index=False)
print(f"Saved summary to: {summary_path}")

## Output structure

```text
OUTPUT_ROOT/
└── causalman_<scale>_<timestamp>/
    ├── generation_summary.csv
    ├── seed_004/
    │   ├── causalman_<scale>_observational_n<N>_RS4.csv
    │   ├── observable_variables.txt
    │   ├── projected_ground_truth_admg.graphml
    │   ├── projected_ground_truth_mag.graphml
    │   ├── causalman_<scale>_interventional_table.csv
    │   └── simulator_output/
    └── ...
```